# Final-B 17-class — Recompute all metrics

Re-evaluate all 30 finalized checkpoints on the validation and test splits. The notebook rebuilds aggregate, per-class, confusion-matrix, prediction, main multi-seed, and ablation artifacts directly from `best.pth`.

In [1]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = Path.cwd()

if not (ROOT / "src").exists():
    ROOT = next(
        (parent for parent in [ROOT, *ROOT.parents] if (parent / "src").exists()),
        ROOT,
    )

sys.path.insert(0, str(ROOT))
print(f"ROOT: {ROOT}")

ROOT: /home/sakana/Code/ManifoldsTeam/woodbench


In [ ]:
# if Path('/content').exists():
#     %pip install -q -r {ROOT}/requirements.txt

#     DATA_DIR = ROOT / "data" / "task-B-group-split"
#     if not DATA_DIR.exists():
#         from huggingface_hub import snapshot_download

#         snapshot_download(
#             repo_id="ny0g507h3p/wood-task-B",
#             repo_type="dataset",
#             local_dir=str(DATA_DIR),
#             ignore_patterns=["*.md", "README*"],
#         )

In [2]:
import gc
import json
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    log_loss,
    matthews_corrcoef,
    precision_recall_fscore_support,
    top_k_accuracy_score,
)
from torch.utils.data import DataLoader

from src.data import FlatConiferDataset
from src.data.transforms import get_val_transforms
from src.models import create_model

DATA_DIR = ROOT / "data" / "task-B-group-split"
EXPERIMENT_DIR = ROOT / "data" / "experiments" / "Task-B"
OUTPUT_DIR = ROOT / "data" / "experiments" / "phase_1" / "final_b_recomputed"

BACKBONES = [
    "resnet50",
    "efficientnet_b3",
    "mobilenetv3_large",
    "convnext_tiny",
    "swin_tiny",
]
MAIN_SEEDS = [42, 123, 456]
ABLATIONS = [
    "taskB_90rotation",
    "taskB_label_smoothing_0.0",
    "taskB_label_smoothing_0.2",
]
EXCLUDED_CLASSES = {
    "Calocedrus sp",
    "Cupressus funebris (Cupressus pendula)",
    "Pinus sp",
}
EXPECTED_SPLIT_COUNTS = {"train": 3365, "val": 903, "test": 532}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
BATCH_SIZE = 64
NUM_WORKERS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")
print(f"Checkpoints: {EXPERIMENT_DIR}")
print(f"Output: {OUTPUT_DIR}")

Device: cuda
GPU: NVIDIA GeForce RTX 3060
Checkpoints: /home/sakana/Code/ManifoldsTeam/woodbench/data/experiments/Task-B
Output: /home/sakana/Code/ManifoldsTeam/woodbench/data/experiments/phase_1/final_b_recomputed


In [3]:
def count_images(path: Path) -> int:
    return sum(
        item.is_file() and item.suffix.lower() in IMAGE_EXTENSIONS
        for item in path.rglob("*")
    )


def validate_dataset(data_dir: Path) -> list[str]:
    split_classes = {}
    for split, expected_count in EXPECTED_SPLIT_COUNTS.items():
        split_dir = data_dir / split
        if not split_dir.exists():
            raise FileNotFoundError(f"Missing Final-B split: {split_dir}")
        classes = sorted(path.name for path in split_dir.iterdir() if path.is_dir())
        actual_count = count_images(split_dir)
        assert len(classes) == 17, f"{split}: expected 17 classes, got {len(classes)}"
        assert EXCLUDED_CLASSES.isdisjoint(classes), f"{split}: excluded class remains"
        assert actual_count == expected_count, (
            f"{split}: expected {expected_count} images, got {actual_count}"
        )
        split_classes[split] = classes

    reference = split_classes["train"]
    for split in ["val", "test"]:
        assert split_classes[split] == reference, f"{split}: class mapping differs from train"
    return reference


def discover_checkpoints(experiment_dir: Path) -> pd.DataFrame:
    records = []
    for checkpoint in sorted(experiment_dir.rglob("*_best.pth")):
        parts = checkpoint.relative_to(experiment_dir).parts
        if parts[0] == "main":
            experiment = "main"
            seed = int(parts[1].removeprefix("seed_"))
            backbone = parts[2]
        else:
            experiment = parts[0]
            seed = 42
            backbone = parts[1]
        records.append(
            {
                "experiment": experiment,
                "seed": seed,
                "backbone": backbone,
                "checkpoint": checkpoint,
            }
        )

    inventory = pd.DataFrame(records)
    expected = {
        *(f"main|{seed}|{backbone}" for seed in MAIN_SEEDS for backbone in BACKBONES),
        *(f"{experiment}|42|{backbone}" for experiment in ABLATIONS for backbone in BACKBONES),
    }
    actual = {
        f"{row.experiment}|{row.seed}|{row.backbone}"
        for row in inventory.itertuples(index=False)
    }
    assert len(inventory) == 30, f"Expected 30 checkpoints, got {len(inventory)}"
    assert actual == expected, (
        f"Checkpoint inventory mismatch. Missing={sorted(expected - actual)}, "
        f"extra={sorted(actual - expected)}"
    )
    return inventory.sort_values(["experiment", "seed", "backbone"]).reset_index(drop=True)


class_names = validate_dataset(DATA_DIR)
checkpoint_inventory = discover_checkpoints(EXPERIMENT_DIR)
print(f"Dataset validated: {len(class_names)} classes")
print(f"Checkpoint inventory validated: {len(checkpoint_inventory)} best checkpoints")
checkpoint_inventory[["experiment", "seed", "backbone"]]

Dataset validated: 17 classes
Checkpoint inventory validated: 30 best checkpoints


,experiment,seed,backbone
0,main,42,convnext_tiny
1,main,42,efficientnet_b3
2,main,42,mobilenetv3_large
3,main,42,resnet50
4,main,42,swin_tiny
5,main,123,convnext_tiny
6,main,123,efficientnet_b3
7,main,123,mobilenetv3_large
8,main,123,resnet50
9,main,123,swin_tiny


In [4]:
transform = get_val_transforms(img_size=224)
datasets = {
    split: FlatConiferDataset(
        DATA_DIR,
        split=split,
        transform=transform,
        task="taskB",
        classes=class_names,
    )
    for split in ["val", "test"]
}
loaders = {
    split: DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
    )
    for split, dataset in datasets.items()
}


def load_checkpoint(path: Path):
    try:
        payload = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        payload = torch.load(path, map_location="cpu")
    return payload, payload.get("model_state_dict", payload)


def checkpoint_metadata(payload: dict) -> dict:
    metrics = payload.get("metrics", {})
    best_epoch = payload.get("best_epoch", payload.get("epoch", -1))
    return {
        "checkpoint_best_val_acc": float(
            payload.get("best_val_acc", metrics.get("val_acc", np.nan))
        ),
        "checkpoint_best_epoch": int(best_epoch) + 1,
        "checkpoint_saved_epoch": int(payload.get("epoch", -1)) + 1,
    }


# Preflight: strict-load one checkpoint and validate one forward batch.
preflight_record = checkpoint_inventory.iloc[0]
preflight_payload, preflight_state = load_checkpoint(preflight_record.checkpoint)
preflight_model = create_model(
    preflight_record.backbone, num_classes=len(class_names), pretrained=False
).to(DEVICE)
preflight_model.load_state_dict(preflight_state, strict=True)
preflight_model.eval()
preflight_images, _ = next(iter(loaders["val"]))
with torch.inference_mode():
    preflight_logits = preflight_model(preflight_images.to(DEVICE))
assert preflight_logits.shape == (len(preflight_images), len(class_names))
print(f"Preflight passed: logits={tuple(preflight_logits.shape)}")
del preflight_model, preflight_payload, preflight_state, preflight_images, preflight_logits
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

Preflight passed: logits=(64, 17)


In [5]:
@torch.inference_mode()
def infer(model: torch.nn.Module, loader: DataLoader):
    labels, predictions, probabilities = [], [], []
    for images, targets in loader:
        logits = model(images.to(DEVICE, non_blocking=True))
        probs = torch.softmax(logits, dim=1)
        labels.append(targets.numpy())
        predictions.append(probs.argmax(dim=1).cpu().numpy())
        probabilities.append(probs.cpu().numpy())
    return np.concatenate(labels), np.concatenate(predictions), np.concatenate(probabilities)


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray):
    label_ids = np.arange(len(class_names))
    aggregate = {
        "log_loss": float(log_loss(y_true, y_prob, labels=label_ids)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "matthews_corrcoef": float(matthews_corrcoef(y_true, y_pred)),
        "cohen_kappa": float(cohen_kappa_score(y_true, y_pred)),
        "top3_accuracy": float(
            top_k_accuracy_score(y_true, y_prob, k=3, labels=label_ids)
        ),
    }
    for average in ["macro", "micro", "weighted"]:
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=label_ids,
            average=average,
            zero_division=0,
        )
        aggregate[f"precision_{average}"] = float(precision)
        aggregate[f"recall_{average}"] = float(recall)
        aggregate[f"f1_{average}"] = float(f1)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=label_ids, average=None, zero_division=0
    )
    per_class = pd.DataFrame(
        {
            "class_index": label_ids,
            "class_name": class_names,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support.astype(int),
        }
    )
    matrix = confusion_matrix(y_true, y_pred, labels=label_ids)
    return aggregate, per_class, matrix


def prediction_table(dataset, y_true, y_pred, y_prob) -> pd.DataFrame:
    image_paths = [str(path.relative_to(ROOT)) for path, _ in dataset.samples]
    top3_indices = np.argsort(y_prob, axis=1)[:, -3:][:, ::-1]
    return pd.DataFrame(
        {
            "image_path": image_paths,
            "true_index": y_true,
            "true_class": [class_names[index] for index in y_true],
            "predicted_index": y_pred,
            "predicted_class": [class_names[index] for index in y_pred],
            "confidence": y_prob.max(axis=1),
            "correct": y_true == y_pred,
            "top3_classes": [
                " | ".join(class_names[index] for index in indices)
                for indices in top3_indices
            ],
        }
    )

In [6]:
run_rows = []
per_class_frames = []
prediction_frames = []
confusion_matrices = {}

for record in checkpoint_inventory.itertuples(index=False):
    payload, state_dict = load_checkpoint(record.checkpoint)
    model = create_model(record.backbone, num_classes=len(class_names), pretrained=False)
    model.load_state_dict(state_dict, strict=True)
    model = model.to(DEVICE).eval()
    run_metadata = checkpoint_metadata(payload)

    for split, loader in loaders.items():
        y_true, y_pred, y_prob = infer(model, loader)
        aggregate, per_class, matrix = compute_metrics(y_true, y_pred, y_prob)
        run_id = f"{record.experiment}__seed_{record.seed}__{record.backbone}__{split}"
        confusion_key = run_id.replace(".", "p")
        common = {
            "run_id": run_id,
            "experiment": record.experiment,
            "seed": int(record.seed),
            "backbone": record.backbone,
            "split": split,
            "num_classes": len(class_names),
            "num_images": len(y_true),
            "checkpoint": str(record.checkpoint.relative_to(ROOT)),
            "confusion_matrix_key": confusion_key,
        }
        run_rows.append({**common, **run_metadata, **aggregate})
        per_class_frames.append(per_class.assign(**common))
        prediction_frames.append(
            prediction_table(datasets[split], y_true, y_pred, y_prob).assign(**common)
        )
        confusion_matrices[confusion_key] = matrix
        print(
            f"{record.experiment:<27s} | seed={record.seed:<3d} | "
            f"{record.backbone:<20s} | {split:<4s} | "
            f"Acc={aggregate['accuracy'] * 100:6.2f}% | "
            f"F1={aggregate['f1_macro'] * 100:6.2f}%"
        )

    del model, payload, state_dict
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

run_metrics = pd.DataFrame(run_rows)
per_class_metrics = pd.concat(per_class_frames, ignore_index=True)
predictions = pd.concat(prediction_frames, ignore_index=True)
assert len(run_metrics) == 60
run_metrics

main                        | seed=42  | convnext_tiny        | val  | Acc= 99.89% | F1= 99.90%
main                        | seed=42  | convnext_tiny        | test | Acc= 99.62% | F1= 99.61%
main                        | seed=42  | efficientnet_b3      | val  | Acc= 95.79% | F1= 95.86%
main                        | seed=42  | efficientnet_b3      | test | Acc= 96.99% | F1= 96.95%
main                        | seed=42  | mobilenetv3_large    | val  | Acc= 95.24% | F1= 95.05%
main                        | seed=42  | mobilenetv3_large    | test | Acc= 98.50% | F1= 98.55%
main                        | seed=42  | resnet50             | val  | Acc= 90.59% | F1= 89.24%
main                        | seed=42  | resnet50             | test | Acc= 94.74% | F1= 94.87%
main                        | seed=42  | swin_tiny            | val  | Acc= 99.67% | F1= 99.68%
main                        | seed=42  | swin_tiny            | test | Acc= 99.06% | F1= 99.13%
main                        | seed=123 |

,run_id,experiment,seed,backbone,split,num_classes,num_images,checkpoint,confusion_matrix_key,checkpoint_best_val_acc,...,top3_accuracy,precision_macro,recall_macro,f1_macro,precision_micro,recall_micro,f1_micro,precision_weighted,recall_weighted,f1_weighted
0,main__seed_42__convnext_tiny__val,main,42,convnext_tiny,val,17,903,data/experiments/Task-B/main/seed_42/convnext_...,main__seed_42__convnext_tiny__val,99.889258,...,1.000000,0.998930,0.999020,0.998966,0.998893,0.998893,0.998893,0.998913,0.998893,0.998893
1,main__seed_42__convnext_tiny__test,main,42,convnext_tiny,test,17,532,data/experiments/Task-B/main/seed_42/convnext_...,main__seed_42__convnext_tiny__test,99.889258,...,1.000000,0.996379,0.995915,0.996081,0.996241,0.996241,0.996241,0.996356,0.996241,0.996237
2,main__seed_42__efficientnet_b3__val,main,42,efficientnet_b3,val,17,903,data/experiments/Task-B/main/seed_42/efficient...,main__seed_42__efficientnet_b3__val,95.791805,...,0.998893,0.962623,0.959388,0.958643,0.957918,0.957918,0.957918,0.961196,0.957918,0.956992
3,main__seed_42__efficientnet_b3__test,main,42,efficientnet_b3,test,17,532,data/experiments/Task-B/main/seed_42/efficient...,main__seed_42__efficientnet_b3__test,95.791805,...,1.000000,0.971164,0.970001,0.969505,0.969925,0.969925,0.969925,0.971957,0.969925,0.969852
4,main__seed_42__mobilenetv3_large__val,main,42,mobilenetv3_large,val,17,903,data/experiments/Task-B/main/seed_42/mobilenet...,main__seed_42__mobilenetv3_large__val,95.348837,...,0.996678,0.957929,0.949933,0.950524,0.952381,0.952381,0.952381,0.956272,0.952381,0.951167
5,main__seed_42__mobilenetv3_large__test,main,42,mobilenetv3_large,test,17,532,data/experiments/Task-B/main/seed_42/mobilenet...,main__seed_42__mobilenetv3_large__test,95.348837,...,1.000000,0.986688,0.985263,0.985533,0.984962,0.984962,0.984962,0.985764,0.984962,0.984910
6,main__seed_42__resnet50__val,main,42,resnet50,val,17,903,data/experiments/Task-B/main/seed_42/resnet50/...,main__seed_42__resnet50__val,90.586932,...,0.978959,0.918979,0.896530,0.892394,0.905869,0.905869,0.905869,0.924182,0.905869,0.901454
7,main__seed_42__resnet50__test,main,42,resnet50,test,17,532,data/experiments/Task-B/main/seed_42/resnet50/...,main__seed_42__resnet50__test,90.586932,...,0.998120,0.953093,0.948177,0.948664,0.947368,0.947368,0.947368,0.949938,0.947368,0.946764
8,main__seed_42__swin_tiny__val,main,42,swin_tiny,val,17,903,data/experiments/Task-B/main/seed_42/swin_tiny...,main__seed_42__swin_tiny__val,99.667774,...,1.000000,0.997012,0.996709,0.996825,0.996678,0.996678,0.996678,0.996766,0.996678,0.996684
9,main__seed_42__swin_tiny__test,main,42,swin_tiny,test,17,532,data/experiments/Task-B/main/seed_42/swin_tiny...,main__seed_42__swin_tiny__test,99.667774,...,1.000000,0.992051,0.991830,0.991348,0.990602,0.990602,0.990602,0.991872,0.990602,0.990591


In [7]:
METRIC_COLUMNS = [
    "log_loss",
    "accuracy",
    "balanced_accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "precision_micro",
    "recall_micro",
    "f1_micro",
    "precision_weighted",
    "recall_weighted",
    "f1_weighted",
    "matthews_corrcoef",
    "cohen_kappa",
    "top3_accuracy",
]

main_test = run_metrics.query("experiment == 'main' and split == 'test'")
summary_rows = []
for backbone, group in main_test.groupby("backbone", sort=False):
    summary = {"backbone": backbone, "runs": len(group)}
    for metric in METRIC_COLUMNS:
        summary[f"{metric}_mean"] = group[metric].mean()
        summary[f"{metric}_std"] = group[metric].std(ddof=0)
    summary_rows.append(summary)
main_test_summary = pd.DataFrame(summary_rows)

baseline = main_test.query("seed == 42")[["backbone", *METRIC_COLUMNS]].rename(
    columns={metric: f"main_seed42_{metric}" for metric in METRIC_COLUMNS}
)
ablation_comparison = run_metrics.query(
    "experiment != 'main' and split == 'test'"
).merge(baseline, on="backbone", validate="many_to_one")
for metric in METRIC_COLUMNS:
    ablation_comparison[f"{metric}_delta_vs_main"] = (
        ablation_comparison[metric] - ablation_comparison[f"main_seed42_{metric}"]
    )

metadata = {
    "recomputed_at_utc": datetime.now(timezone.utc).isoformat(),
    "device_type": DEVICE.type,
    "device_name": (
        torch.cuda.get_device_name(DEVICE)
        if DEVICE.type == "cuda"
        else platform.processor() or platform.machine()
    ),
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "classes": class_names,
    "excluded_classes": sorted(EXCLUDED_CLASSES),
    "split_counts": EXPECTED_SPLIT_COUNTS,
    "checkpoints": len(checkpoint_inventory),
    "evaluated_splits": ["val", "test"],
    "metric_columns": METRIC_COLUMNS,
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
inventory_to_save = checkpoint_inventory.assign(
    checkpoint=checkpoint_inventory["checkpoint"].map(lambda path: str(path.relative_to(ROOT)))
)
inventory_to_save.to_csv(OUTPUT_DIR / "checkpoint_inventory.csv", index=False)
run_metrics.to_csv(OUTPUT_DIR / "split_metrics.csv", index=False)
run_metrics.to_json(
    OUTPUT_DIR / "split_metrics.json", orient="records", indent=2, force_ascii=False
)
per_class_metrics.to_csv(OUTPUT_DIR / "per_class_metrics.csv", index=False)
predictions.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
main_test_summary.to_csv(OUTPUT_DIR / "main_test_summary.csv", index=False)
ablation_comparison.to_csv(OUTPUT_DIR / "ablation_vs_main_seed42.csv", index=False)
np.savez_compressed(OUTPUT_DIR / "confusion_matrices.npz", **confusion_matrices)
(OUTPUT_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(f"Saved all recomputed artifacts to: {OUTPUT_DIR}")
main_test_summary[[
    "backbone",
    "accuracy_mean",
    "accuracy_std",
    "f1_macro_mean",
    "f1_macro_std",
]].assign(
    accuracy_mean=lambda frame: frame.accuracy_mean * 100,
    accuracy_std=lambda frame: frame.accuracy_std * 100,
    f1_macro_mean=lambda frame: frame.f1_macro_mean * 100,
    f1_macro_std=lambda frame: frame.f1_macro_std * 100,
).round(3)

Saved all recomputed artifacts to: /home/sakana/Code/ManifoldsTeam/woodbench/data/experiments/phase_1/final_b_recomputed


,backbone,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std
0,convnext_tiny,99.499,0.089,99.524,0.059
1,efficientnet_b3,97.118,0.177,97.138,0.221
2,mobilenetv3_large,98.371,0.319,98.407,0.339
3,resnet50,93.672,1.022,93.844,0.998
4,swin_tiny,99.436,0.307,99.481,0.282
